In [ ]:
#AUTO-RUN-FIRST (Colab): fetch repo so data/ is present. Safe to re-run.
import os, subprocess
from pathlib import Path
if not Path("data/labels.jsonl").exists():
    if not Path("bench").exists():
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/hsmahon/eval-decision-models-captcha.git",
                        "bench"], check=True)
    os.chdir("bench")
print("cwd:", Path.cwd())
print("labels present:", Path("data/labels.jsonl").exists())

# Vision Decision Eval — strands-decider-2B-hobson-v21

**Question:** How does a small vision decision model perform as visual decision complexity increases?

Benchmark: 150 synthetic CAPTCHA-style grids (30 each of 3x3, 3x4, 4x4, 4x5, 5x5; 10 easy / 10 medium / 10 hard per grid). Task per cell: `Is a stoplight visible in this image?`

This notebook is Artifact 1. It produces `data/results.json`, which `index.html` (Artifact 2) renders. The webpage never calls the model.

> Ethics: synthetic benchmark only, not for bypassing real CAPTCHAs.

## 0. Setup — Colab or local Mac/Linux
Only 2 packages. Vision needs `transformers>=5.18` (via the `[vision]` extra).

In [ ]:
# Colab: uncomment next line
# !pip install -q "strands-decider[vision] @ git+https://github.com/strands-labs/strands-decider.git" pillow
# NOTE: vision lives on git main only — PyPI 0.1.0 has no [vision] extra and no vision code.

import json, time, math, statistics, platform, datetime, subprocess
from pathlib import Path
from PIL import Image

ROOT = Path.cwd()
DATA = ROOT / "data"
assert (DATA / "labels.jsonl").exists(), "missing data/labels.jsonl -- copy data/ next to eval.ipynb"
print("python:", platform.python_version(), "| pillow:", Image.__version__)
try:
    import transformers; print("transformers:", transformers.__version__)
except Exception as e:
    print("transformers not installed yet:", e)
try:
    import strands_decider; print("strands_decider: installed")
except Exception as e:
    print("strands_decider not installed yet:", e)

## 1. Dataset
`labels.jsonl` holds cell-level ground truth (`positive_cells[]` + per-cell `label`). Fixed artifact — no image generation during eval.

In [ ]:
rows = [json.loads(l) for l in open(DATA / "labels.jsonl")]
print("images:", len(rows))
from collections import Counter
print("by grid:", dict(Counter(r["grid"] for r in rows)))
print("by difficulty:", dict(Counter(r["difficulty"] for r in rows)))
print("total cell decisions (full run):", sum(r["rows"] * r["cols"] for r in rows))
print(json.dumps(rows[0], indent=1)[:500])
for r in rows[:3]:
    p = DATA / r["image"]
    print(r["id"], r["grid"], r["difficulty"], "positives:", r["positive_cells"], "exists:", p.exists())
    display(Image.open(p).resize((300, 300), Image.NEAREST))

## 2. Model — one `noul` question per cell crop
Model: `StrandsAgents/strands-decider-2B-hobson-v21`. Vision = kept Qwen3.5 vision tower (frozen, no vision training — see upstream `docs/vision.md`). Install from git main: `pip install "strands-decider[vision] @ git+https://github.com/strands-labs/strands-decider.git"` (PyPI 0.1.0 is text-only). Fixed prompt-v2 + criteria for every cell. Python: `load_vision_engine(...)` → `engine.ask_images(state, questions, images=[b64])`; CLI: `strands-decider ask MODEL --state "" --image crop.png --noul "..."`.

In [ ]:
MODEL_ID = "StrandsAgents/strands-decider-2B-hobson-v21"
PROMPT = "Is a stoplight visible in this image?"
PROMPT_VERSION = "prompt-v2-neutral"
CRITERIA = {"true": "A traffic stoplight is visible, whole or partial — including cropped edges, night shots, or small/distant signals",
            "false": "No stoplight visible — including lookalikes such as street lamps, car tail lights, or traffic signs without lights"}

# Vision ships on git main only (PyPI 0.1.0 is text-only):
#   pip install "strands-decider[vision] @ git+https://github.com/strands-labs/strands-decider.git"
# Python API (see docs/vision.md upstream): load_vision_engine(...) -> engine.ask_images(state, questions, images=[b64...])
def load_model():
    try:
        from strands_decider.vision import load_vision_engine
        return ("VisionEngine", load_vision_engine(MODEL_ID))
    except Exception as e:
        print("vision engine load failed:", str(e)[:300])
    print("No vision engine. Install from git main (see above), or use CLI: strands-decider ask MODEL --state '' --image FILE --noul \"...\"")
    return (None, None)

KIND, ENGINE = load_model()
print("backend:", KIND)

def make_questions():
    from strands_decider.schema import NoulQuestion
    return {"stoplight": NoulQuestion(instructions=PROMPT + " Answer based only on visible pixels.",
                                      criteria={"true": CRITERIA["true"], "false": CRITERIA["false"]})}

QUESTIONS = make_questions() if KIND else None

def crop_cells(img_path, rows_n, cols_n):
    im = Image.open(img_path).convert("RGB")
    W, H = im.size
    cw, ch = W // cols_n, H // rows_n
    out = []
    for r in range(rows_n):
        for c in range(cols_n):
            x1, y1 = c * cw, r * ch
            x2 = (c + 1) * cw if c < cols_n - 1 else W
            y2 = (r + 1) * ch if r < rows_n - 1 else H
            out.append(im.crop((x1, y1, x2, y2)))
    return out

def pil_to_b64(pil_img):
    import io, base64
    buf = io.BytesIO()
    pil_img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("ascii")

def ask_cell(crop):
    import time
    t0 = time.perf_counter()
    resp = ENGINE.ask_images("", QUESTIONS, images=[pil_to_b64(crop)])
    ans = resp.answers["stoplight"]
    p = float(ans.noul)  # NoulAnswer has no separate confidence: p IS the uncertainty
    pred = p >= 0.5
    return pred, (p if pred else 1 - p), ("noul=%.4f" % p)


## 3. Run eval
`LIMIT=5` smoke-tests 5 images. Full run = 150 images / 2,460 calls (~8 min at ~200ms). Latency = preprocess + inference via `perf_counter` (no frontend time).

In [ ]:
LIMIT = 5  # None = full 150-image run
DRY_RUN_NO_MODEL = (KIND is None)  # True = cropping/timing check only (pred=False)
if DRY_RUN_NO_MODEL and LIMIT is None:
    raise RuntimeError("model did not load (KIND is None) — refusing a 150-image dry run that would masquerade as results. Fix the install/load in §2, or set LIMIT=5 for a cropping check.")
decisions = []
items = rows if LIMIT is None else rows[:LIMIT]
for r in items:
    pos = set(r["positive_cells"])
    for i, crop in enumerate(crop_cells(DATA / r["image"], r["rows"], r["cols"])):
        t0 = time.perf_counter()
        if DRY_RUN_NO_MODEL:
            pred, conf, raw = False, None, "dry-run"
        else:
            pred, conf, raw = ask_cell(crop)
        dt = (time.perf_counter() - t0) * 1000
        decisions.append({"image_id": r["id"], "grid": r["grid"], "rows": r["rows"], "cols": r["cols"], "difficulty": r["difficulty"], "cell_index": i, "ground_truth": (i in pos), "prediction": bool(pred), "confidence": conf, "latency_ms": round(dt, 1), "model": MODEL_ID, "prompt": PROMPT, "raw": raw})
print(len(decisions), "decisions")
if decisions:
    print("acc:", round(sum(d["prediction"] == d["ground_truth"] for d in decisions) / len(decisions), 3))

## 4. Metrics (stdlib only) -> `data/results.json`
Primary = per-cell accuracy. Secondary = positive-class F1, confidence buckets, mean/median/p95 latency, Wilson 95% CI. Per-image full-success is secondary (falls with grid size even if cell-acc is flat).

In [ ]:
def wilson(p, n, z=1.96):
    if not n: return [0.0, 0.0]
    d = 1 + z*z/n; c = p + z*z/(2*n); m = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return [round(max(0, (c-m)/d), 4), round(min(1, (c+m)/d), 4)]

def summarize(ds):
    n = len(ds)
    acc = sum(d["prediction"] == d["ground_truth"] for d in ds) / max(1, n)
    tp = sum(d["prediction"] and d["ground_truth"] for d in ds)
    fp = sum(d["prediction"] and not d["ground_truth"] for d in ds)
    fn = sum((not d["prediction"]) and d["ground_truth"] for d in ds)
    prec = tp / max(1, tp + fp); rec = tp / max(1, tp + fn)
    f1 = 2*prec*rec / max(1e-9, prec + rec)
    lat = sorted(d["latency_ms"] for d in ds)
    confs = [d["confidence"] for d in ds if d["confidence"] is not None]
    buckets = {}
    for lo in [0, 20, 40, 60, 80]:
        b = [d for d in ds if d["confidence"] is not None and (lo/100 <= d["confidence"] < (lo+20)/100 or (lo == 80 and d["confidence"] == 1.0))]
        buckets[f"{lo}-{lo+20}"] = {"n": len(b), "acc": (round(sum(d["prediction"] == d["ground_truth"] for d in b)/len(b), 4) if b else None)}
    return {"n_images": len(set(d["image_id"] for d in ds)), "n_cells": n, "accuracy": round(acc, 4), "ci95": wilson(acc, n), "precision_pos": round(prec, 4), "recall_pos": round(rec, 4), "f1_pos": round(f1, 4), "avg_confidence": (round(sum(confs)/len(confs), 4) if confs else None), "lat_mean": (round(statistics.mean(lat), 1) if lat else None), "lat_median": (round(statistics.median(lat), 1) if lat else None), "lat_p95": (round(lat[min(len(lat)-1, int(0.95*len(lat)))], 1) if lat else None), "conf_buckets": buckets}

by_grid = {g: summarize([d for d in decisions if d["grid"] == g]) for g in ["3x3", "3x4", "4x4", "4x5", "5x5"]}
by_gd = {f"{g}/{df}": summarize([d for d in decisions if d["grid"] == g and d["difficulty"] == df]) for g in ["3x3", "3x4", "4x4", "4x5", "5x5"] for df in ["easy", "medium", "hard"]}
per_image = []
for r in items:
    ds = [d for d in decisions if d["image_id"] == r["id"]]
    s = summarize(ds) if ds else {"accuracy": None, "avg_confidence": None}
    per_image.append({"image_id": r["id"], "grid": r["grid"], "difficulty": r["difficulty"], "rows": r["rows"], "cols": r["cols"], "cell_acc": s.get("accuracy"), "full_success": (s.get("accuracy") == 1.0) if s.get("accuracy") is not None else None, "avg_conf": s.get("avg_confidence"), "total_ms": round(sum(d["latency_ms"] for d in ds), 1) if ds else None, "n_cells": len(ds), "positive_cells": r["positive_cells"]})
import importlib.metadata as md
ver = lambda p: md.version(p) if True else None
def _ver(p):
    try: return md.version(p)
    except Exception: return None
results = {"schema_version": 1, "manifest": {"model": MODEL_ID, "prompt": PROMPT, "prompt_version": PROMPT_VERSION, "criteria": CRITERIA, "est_cost_usd": 0.0, "temperature": "n/a (decision head)", "benchmark": "captcha_vision_benchmark_150 v1", "n_images_run": len(items), "n_images_total": 150, "full_run": LIMIT is None, "backend": KIND, "timestamp": datetime.datetime.utcnow().isoformat() + "Z", "packages": {"strands-decider": _ver("strands-decider"), "transformers": _ver("transformers"), "pillow": _ver("Pillow")}}, "overall": summarize(decisions), "by_grid": by_grid, "by_grid_difficulty": by_gd, "per_image": sorted(per_image, key=lambda x: (x["cell_acc"] is None, x["cell_acc"] or 0)), "decisions": decisions}
open(DATA / "results.json", "w").write(json.dumps(results, indent=1))
print("wrote data/results.json:", len(decisions), "decisions")
print(json.dumps({"overall": results["overall"], "by_grid": {k: v["accuracy"] for k, v in by_grid.items()}}, indent=1))
print("Open index.html -- it reads ./data/results.json (no model call in page).")

## 5. Second backend — Liquid d1-3B
Same 150 grids, same prompt-v2 + criteria, same schema — only the model call changes. d1-3B has a native calibrated `noul` head (`system_one`, zero output tokens). Needs `transformers>=5.14` + torch, bf16 (~7-8 GB, fits T4). Run §4 first (it defines nothing this section needs — the metrics code below is self-contained), or just Run All top to bottom.

In [ ]:
# Colab: uncomment next line (strands §0 already installed torch as a dependency; this ensures version floor)
# !pip install -q "transformers>=5.14" pillow accelerate

D1_ID = "LiquidAI/d1-3B"
try:
    import torch
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
    import torch

def load_d1():
    try:
        from transformers import AutoModel
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        dt = torch.bfloat16 if dev == "cuda" else torch.float32
        m = AutoModel.from_pretrained(D1_ID, trust_remote_code=True, dtype=dt).to(dev).eval()
        return ("d1-system-one", m, dev)
    except Exception as e:
        print("d1 load failed:", str(e)[:300])
        return (None, None, None)

D1_KIND, D1_MODEL, D1_DEVICE = load_d1()
print("backend:", D1_KIND, "| device:", D1_DEVICE)
print("expected: model.system_one(state, questions, images=[...]) -> {'answers': {'stoplight': {'noul': p}}}")

D1_QUESTIONS = {"stoplight": {"type": "noul", "instructions": PROMPT + " Answer based only on visible pixels.",
                              "criteria": {"true": CRITERIA["true"], "false": CRITERIA["false"]}}}

def ask_d1(crop):
    with torch.inference_mode():
        out = D1_MODEL.system_one(None, D1_QUESTIONS, images=[crop])
    ans = out["answers"]["stoplight"]
    p = float(ans.get("noul", ans.get("value")))
    pred = p >= 0.5
    return pred, (p if pred else 1 - p), ("noul=%.4f" % p)

## 6. Run d1 eval
Same `LIMIT` pattern: 5 first (also confirms the `system_one` signature on this `transformers` version — d1-3B is brand new), then `None` for the full 150.

In [ ]:
D1_LIMIT = 5  # None = full 150-image run
d1_decisions = []
d1_items = rows if D1_LIMIT is None else rows[:D1_LIMIT]
for r in d1_items:
    pos = set(r["positive_cells"])
    for i, crop in enumerate(crop_cells(DATA / r["image"], r["rows"], r["cols"])):
        t0 = time.perf_counter()
        if D1_KIND is None:
            pred, conf, raw = False, None, "dry-run"
        else:
            pred, conf, raw = ask_d1(crop)
        dt = (time.perf_counter() - t0) * 1000
        d1_decisions.append({"image_id": r["id"], "grid": r["grid"], "rows": r["rows"], "cols": r["cols"], "difficulty": r["difficulty"], "cell_index": i, "ground_truth": (i in pos), "prediction": bool(pred), "confidence": conf, "latency_ms": round(dt, 1), "model": D1_ID, "prompt": PROMPT, "raw": raw})
print(len(d1_decisions), "decisions")
if d1_decisions:
    print("acc:", round(sum(d["prediction"] == d["ground_truth"] for d in d1_decisions) / len(d1_decisions), 3))

## 7. d1 metrics → `data/results_d1.json`
Identical schema to §4 (local backend: `est_cost_usd: 0.0`).

In [ ]:
def wilson(p, n, z=1.96):
    if not n: return [0.0, 0.0]
    d = 1 + z*z/n; c = p + z*z/(2*n); m = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return [round(max(0, (c-m)/d), 4), round(min(1, (c+m)/d), 4)]

def summarize(ds):
    n = len(ds)
    acc = sum(d["prediction"] == d["ground_truth"] for d in ds) / max(1, n)
    tp = sum(d["prediction"] and d["ground_truth"] for d in ds)
    fp = sum(d["prediction"] and not d["ground_truth"] for d in ds)
    fn = sum((not d["prediction"]) and d["ground_truth"] for d in ds)
    prec = tp / max(1, tp + fp); rec = tp / max(1, tp + fn)
    f1 = 2*prec*rec / max(1e-9, prec + rec)
    lat = sorted(d["latency_ms"] for d in ds)
    confs = [d["confidence"] for d in ds if d["confidence"] is not None]
    buckets = {}
    for lo in [0, 20, 40, 60, 80]:
        b = [d for d in ds if d["confidence"] is not None and (lo/100 <= d["confidence"] < (lo+20)/100 or (lo == 80 and d["confidence"] == 1.0))]
        buckets[f"{lo}-{lo+20}"] = {"n": len(b), "acc": (round(sum(d["prediction"] == d["ground_truth"] for d in b)/len(b), 4) if b else None)}
    return {"n_images": len(set(d["image_id"] for d in ds)), "n_cells": n, "accuracy": round(acc, 4), "ci95": wilson(acc, n), "precision_pos": round(prec, 4), "recall_pos": round(rec, 4), "f1_pos": round(f1, 4), "avg_confidence": (round(sum(confs)/len(confs), 4) if confs else None), "lat_mean": (round(statistics.mean(lat), 1) if lat else None), "lat_median": (round(statistics.median(lat), 1) if lat else None), "lat_p95": (round(lat[min(len(lat)-1, int(0.95*len(lat)))], 1) if lat else None), "conf_buckets": buckets}

by_grid = {g: summarize([d for d in d1_decisions if d["grid"] == g]) for g in ["3x3", "3x4", "4x4", "4x5", "5x5"]}
by_gd = {f"{g}/{df}": summarize([d for d in d1_decisions if d["grid"] == g and d["difficulty"] == df]) for g in ["3x3", "3x4", "4x4", "4x5", "5x5"] for df in ["easy", "medium", "hard"]}
per_image = []
for r in d1_items:
    ds = [d for d in d1_decisions if d["image_id"] == r["id"]]
    s = summarize(ds) if ds else {"accuracy": None, "avg_confidence": None}
    per_image.append({"image_id": r["id"], "grid": r["grid"], "difficulty": r["difficulty"], "rows": r["rows"], "cols": r["cols"], "cell_acc": s.get("accuracy"), "full_success": (s.get("accuracy") == 1.0) if s.get("accuracy") is not None else None, "avg_conf": s.get("avg_confidence"), "total_ms": round(sum(d["latency_ms"] for d in ds), 1) if ds else None, "n_cells": len(ds), "positive_cells": r["positive_cells"]})
import importlib.metadata as md
ver = lambda p: md.version(p) if True else None
def _ver(p):
    try: return md.version(p)
    except Exception: return None
results = {"schema_version": 1, "manifest": {"model": D1_ID, "prompt": PROMPT, "prompt_version": PROMPT_VERSION, "criteria": CRITERIA, "est_cost_usd": 0.0, "temperature": "n/a (decision head)", "benchmark": "captcha_vision_benchmark_150 v1", "n_images_run": len(d1_items), "n_images_total": 150, "full_run": D1_LIMIT is None, "backend": "d1", "timestamp": datetime.datetime.utcnow().isoformat() + "Z", "packages": {"strands-decider": _ver("strands-decider"), "transformers": _ver("transformers"), "pillow": _ver("Pillow")}}, "overall": summarize(d1_decisions), "by_grid": by_grid, "by_grid_difficulty": by_gd, "per_image": sorted(per_image, key=lambda x: (x["cell_acc"] is None, x["cell_acc"] or 0)), "decisions": d1_decisions}
open(DATA / "results_d1.json", "w").write(json.dumps(results, indent=1))
print("wrote data/results_d1.json:", len(d1_decisions), "decisions")
print(json.dumps({"overall": results["overall"], "by_grid": {k: v["accuracy"] for k, v in by_grid.d1_items()}}, indent=1))
print("Open index.html -- it reads ./data/results.json (no model call in page).")